In [25]:
import pandas as pd
import numpy as np
import re  # Regular expressions to clean leading zeros
import ast  # To safely evaluate string representations of lists

# Read the data and skip the first row
df_small = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/data/bin>50.txt', sep='|', skipinitialspace=True)
df_small = df_small.iloc[1:]  # Skip the first row

# Clean column names and data
df_small.columns = df_small.columns.str.strip()
for col in df_small.columns:
    df_small[col] = df_small[col].str.strip() if df_small[col].dtype == 'object' else df_small[col]


In [26]:
df_small.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1322 entries, 1 to 1322
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       0 non-null      float64
 1   compound_splash  1322 non-null   object 
 2   splash_list      1322 non-null   object 
 3   sample_list      1322 non-null   object 
 4   mass_list        1322 non-null   object 
 5   ri_list          1322 non-null   object 
 6   rt_list          1322 non-null   object 
 7   Unnamed: 7       0 non-null      float64
dtypes: float64(2), object(6)
memory usage: 82.8+ KB


In [27]:

# 2. Bootstrap Confidence Interval Function
def bootstrap_ci(data, statistic_func, n_bootstrap=1000, ci=95):
    bootstrapped_stats = []
    n = len(data)
    for _ in range(n_bootstrap):
        resampled = np.random.choice(data, size=n, replace=True)
        bootstrapped_stats.append(statistic_func(resampled))

    # Percentiles for CI
    lower_bound = np.percentile(bootstrapped_stats, (100 - ci) / 2)
    upper_bound = np.percentile(bootstrapped_stats, 100 - (100 - ci) / 2)
    return lower_bound, upper_bound


In [28]:
# 3. Compute Bootstrap CIs for Each Bin
results = []
for _, row in df_small.iterrows():
    # Convert the string representation of lists into actual arrays
    mass_data = np.array(list(ast.literal_eval(row["mass_list"])), dtype=float)
    
    ri_data = np.array(list(ast.literal_eval(row["ri_list"])), dtype=float)
    
    # Calculate bootstrap confidence intervals for mass and RI
    mass_mean_ci = bootstrap_ci(mass_data, np.mean)
    mass_std_ci = bootstrap_ci(mass_data, np.std)
    ri_mean_ci = bootstrap_ci(ri_data, np.mean)
    ri_std_ci = bootstrap_ci(ri_data, np.std)
    
    # Store the results in a dictionary
    results.append({
    "compound_splash": row["compound_splash"],
    "mass_mean_ci": mass_mean_ci,
    "mass_std_ci": mass_std_ci, 
    "ri_mean_ci": ri_mean_ci,
    "ri_std_ci": ri_std_ci,
    "n_samples": len(ri_data)  # Add sample size column
})

# 4. Create Results DataFrame
results_df = pd.DataFrame(results)

                                    compound_splash  \
0     splash10-0002-0090000000-3b4b9cc16e91ef0eee7e   
1     splash10-0002-0090000000-d3e9d82e926125d30c16   
2     splash10-0002-0453090000-514d74b60cd1e64e43c9   
3     splash10-0002-0490000000-2286dd4bd4a445e6aea0   
4     splash10-0002-0900000000-0d346cf5bf0ac8114ef6   
...                                             ...   
1317  splash10-0zg0-5940000000-d84c30d5f60023b01b93   
1318  splash10-0zgi-0930000000-ea6cb4818527c32e89f3   
1319  splash10-0zi3-8930000000-990a5d4eb7e6f65d8273   
1320  splash10-1001-3920000000-b8251350436b3a9b9af5   
1321  splash10-11b9-9221101000-b7feee0ff62ff29d3840   

                                  mass_mean_ci  \
0     (246.90527865122192, 246.90538909628418)   
1       (297.2435580569975, 297.2436842517094)   
2       (595.4944118254003, 595.4946927878286)   
3     (248.07976008030977, 248.08015109597346)   
4       (313.1130669550323, 313.1131292373189)   
...                                    

In [ ]:
def calculate_rt_match_score(experimental_data, reference_data):
    """
    Calculate retention time match score between experimental and reference data
    
    Parameters:
    -----------
    experimental_data : dict
        Contains:
        - rt_mean: mean retention time from bin
        - rt_ci: confidence interval for RT from bootstrap
        - rt_std: standard deviation of RT in bin
    
    reference_data : dict
        Contains:
        - rt: reference retention time from MassWiki
        - rt_tolerance: known tolerance for the method
    
    Returns:
    --------
    float : Score between 0 and 1, where 1 is perfect match
    """
    # 1. Calculate the normalized RT difference
    rt_difference = abs(experimental_data['rt_mean'] - reference_data['rt'])
    
    # 2. Consider the measurement uncertainty from bootstrap
    rt_uncertainty = experimental_data['rt_ci'][1] - experimental_data['rt_ci'][0]
    
    # 3. Consider method-specific tolerance
    total_tolerance = max(rt_uncertainty, reference_data['rt_tolerance'])
    
    # 4. Calculate score using Gaussian function
    # This gives 1.0 for perfect match, declining smoothly as difference increases
    score = np.exp(-(rt_difference**2) / (2 * total_tolerance**2))
    
    return {
        'score': score,
        'details': {
            'rt_difference': rt_difference,
            'uncertainty': rt_uncertainty,
            'tolerance': total_tolerance
        }
    }

# Example usage:
experimental_data = {
    'rt_mean': 24.12,        # From your bin statistics
    'rt_ci': [24.11, 24.13], # From bootstrap analysis
    'rt_std': 0.5886         # From your bin statistics
}

reference_data = {
    'rt': 24.15,            # From MassWiki
    'rt_tolerance': 0.1     # Method-specific tolerance
}

match_result = calculate_rt_match_score(experimental_data, reference_data)